In [2]:
!pip install backports.zstd


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [1]:
from pypaimon import CatalogFactory

catalog_options = {
    'warehouse': 's3://warehouse/paimon',
    'fs.s3.endpoint': 'http://minio:9000',
    'fs.s3.accessKeyId': 'admin',
    'fs.s3.accessKeySecret': 'password',
    'fs.s3.region': 'us-east-1',  # MinIO ignora o valor, mas o campo precisa existir e não vazio
}

catalog = CatalogFactory.create(catalog_options)
table = catalog.get_table('default.credit_scored_paimon')

read_builder = table.new_read_builder()
splits = read_builder.new_scan().plan().splits()
table_read = read_builder.new_read()

# Retorna direto como Arrow Table, fácil de converter pra pandas
arrow_table = table_read.to_arrow(splits)
df = arrow_table.to_pandas()

In [2]:
import pandas as pd

pd.set_option('display.max_colwidth', None)
df.head(10)

,id_cliente,target_real,predicao,justificativa
0,0,1,2,"Saldo negativo, histórico de conta crítica e proposto por um produto relacionado ao consumo (rádio/televisão) com um valor baixo em prazo indicam alto risco."
1,2,1,2,"A falta de conta corrente, histórico de conta crítica com outros bancos, valores altos em prazo longo e propósito educativo não produtivo combinados."
2,3,1,2,"Saldo negativo, histórico de créditos pagos até agora e valor alto em prazo longo indicam alto risco."
3,1,2,2,"Saldo baixo, crédito pago até agora mas propósito de compra de rádio e televisão com um valor alto em prazo longo indicam alto risco."
4,4,2,2,"Saldo negativo, atrasos passados e valor alto em prazo longo indicam alto risco."
5,5,1,2,"Ausência de conta corrente, histórico de créditos pagos até agora e valor alto solicitado com prazo longo (36 meses) indicam alto risco."
6,6,1,2,"Sem conta corrente, histórico de créditos pagos até agora e valor alto em prazo longo indicam alto risco."
7,11,2,2,Saldo negativo e histórico de créditos pagos até agora indicam baixo risco.
8,12,1,1,Saldo baixo e histórico de créditos pagos até agora indicam baixo risco.
9,13,2,2,"Saldo negativo, conta crítica e valor alto em prazo longo indicam alto risco."
